# Notebook Overview (Benzinga → FinBERT → Time-Series Merge)

**Purpose:**
This notebook builds sentiment-augmented equity time series for multiple tickers by (1) downloading historical Benzinga news, (2) scoring each article with FinBERT sentiment probabilities, and (3) aggregating the daily sentiment signals and merging them into the corresponding daily closing-price series.

---

## Data Flow and Outputs

### Phase 1 — Download Benzinga news (JSONL)
For each ticker in `TICKERS`, the notebook queries the Benzinga News API for the full date range **2009-01-01 to 2025-01-31**, split into yearly intervals. Each returned article is:
- converted from HTML to plain text (scripts/styles removed, entities decoded, whitespace normalised),
- sanitized (quotes removed for JSONL robustness),
- written as one JSON object per line to:

`../data/benzinga/{TICKER}.jsonl`

Each line contains:
- `created` (timestamp),
- `title`,
- `body` (cleaned plain text).

---

### Phase 2 — FinBERT sentiment scoring (JSONL)
For each ticker JSONL from Phase 1, the notebook loads **ProsusAI/finbert** once and infers per-article sentiment:

- Input text is truncated to **max 512 tokens** using the FinBERT tokenizer (and the notebook tracks whether truncation occurred).
- The model output logits are converted to probabilities via **softmax**.
- For each article, the notebook writes a JSONL line containing:
  - `created`,
  - `title`,
  - `label` (argmax class),
  - `probs` (dictionary of class → probability)

Output path:

`../data/finbert_data/{TICKER}_finbert_sentiment.jsonl`

The notebook also logs counts for:
- total input lines,
- successfully scored lines,
- ignored lines (runtime errors),
- truncated texts.

---

### Phase 3 — Merge daily sentiment into price time series (CSV)
For each ticker:
1. Load the daily price series from:

`../data/time_series/{ticker.lower()}.csv`
(required columns: `Date`, `Close`)

2. Load the FinBERT JSONL and extract per-article probabilities (`positive`, `negative`, `neutral`) with the article date (`created`).

3. Aggregate sentiment *per day*:
- `pos_mean`: mean positive probability
- `neg_mean`: mean negative probability
- `neut_mean`: mean neutral probability
- `news_count`: number of scored articles that day

4. Restrict the price series to the sentiment date coverage (from first to last sentiment date), then left-join daily sentiment onto daily prices. Missing sentiment days are filled with zeros.

Final output per ticker:

`../data/ts_with_sentiment/{TICKER}_ts_with_sentiment.csv`

Columns:
- `Close`
- `pos_mean`
- `neut_mean`
- `neg_mean`
- `news_count`

---

## Key Assumptions and Notes

- **Time alignment:** all news items are grouped by calendar day using the `created` timestamp; the merged dataset is daily (normalized dates).
- **Coverage restriction:** the price series is truncated to the sentiment range to ensure consistent overlap between modalities.
- **Zero-fill rule:** days without scored news receive `pos_mean=0`, `neut_mean=0`, `neg_mean=0`, `news_count=0`.
- **Reproducibility:** inference is deterministic given fixed model weights; the script logs truncation and error counts to help diagnose data loss.
- **API paging:** for each yearly windo


In [ ]:
# -------- Colab Batch: Benzinga -> FinBERT -> TS Merge (all tickers) --------
import os
import json
from html import unescape
from urllib.parse import unquote
from html.parser import HTMLParser
from datetime import datetime, timedelta
from pathlib import Path
#from google.colab import userdata

import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

from benzinga import news_data
from benzinga.news_data import BadRequestError

In [ ]:
#from google.colab import drive
#drive.mount('/content/drive')

In [ ]:
# =========================
# 0) CONFIG
# =========================
BASE_DIR = "../data"
DATA_DIR = Path(BASE_DIR)

TIME_SERIES_DIR = DATA_DIR / "time_series"
BENZINGA_DIR = DATA_DIR / "benzinga"
FINBERT_DIR = DATA_DIR / "finbert_data"
TS_WITH_SENTIMENT_DIR = DATA_DIR / "ts_with_sentiment"

BENZINGA_DIR.mkdir(parents=True, exist_ok=True)
FINBERT_DIR.mkdir(parents=True, exist_ok=True)
TS_WITH_SENTIMENT_DIR.mkdir(parents=True, exist_ok=True)

TICKERS = [
    "AAPL", "DIS", "IBM", "INTC", "JNJ", "JPM", "KO", "MSFT", "NKE","V"
]

GLOBAL_DATE_FROM = "2009-01-01"
GLOBAL_DATE_TO   = "2025-01-31"

PAGE_SIZE = 100
PAGE_START = 0
PAGE_MAX = 400

MAX_TOKENS = 512
MAX_CHARS_IF_TOO_LONG = 512

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Device: {DEVICE}")

In [ ]:
# =========================
# 1) HELPERS
# =========================
class HTMLTextExtractor(HTMLParser):
    def __init__(self):
        super().__init__()
        self._in_script = False
        self._in_style = False
        self._chunks = []

    def handle_starttag(self, tag, attrs):
        tag = tag.lower()
        if tag == "script":
            self._in_script = True
        elif tag == "style":
            self._in_style = True

    def handle_endtag(self, tag):
        tag = tag.lower()
        if tag == "script":
            self._in_script = False
        elif tag == "style":
            self._in_style = False

    def handle_data(self, data):
        if not self._in_script and not self._in_style:
            self._chunks.append(data)

    def get_text(self) -> str:
        return "".join(self._chunks)


def html_body_to_plain_text(raw_body: str) -> str:
    if not raw_body:
        return ""
    decoded = unquote(unescape(raw_body))
    parser = HTMLTextExtractor()
    parser.feed(decoded)
    return " ".join(parser.get_text().split())


def generate_yearly_ranges(start_date_str: str, end_date_str: str):
    start = datetime.strptime(start_date_str, "%Y-%m-%d").date()
    end = datetime.strptime(end_date_str, "%Y-%m-%d").date()
    ranges = []
    cur = start
    while cur <= end:
        year_end = datetime(cur.year, 12, 31).date()
        cur_end = min(year_end, end)
        ranges.append((cur.strftime("%Y-%m-%d"), cur_end.strftime("%Y-%m-%d")))
        cur = cur_end + timedelta(days=1)
    return ranges


def get_api_key() -> str:
    #api_key = userdata.get("BENZINGA_API_KEY")
    api_key = os.environ.get("BENZINGA_API_KEY")
    if api_key is None:
        raise RuntimeError(
            "BENZINGA_API_KEY not found in Colab Secrets.\n"
            "Go to: Colab → Tools → Secrets → Add new secret\n"
            "Name: BENZINGA_API_KEY"
        )
    print("[INFO] Benzinga API key loaded from Colab Secrets")
    return api_key

def truncate_to_512_tokens(
    text: str,
    tokenizer,
    max_tokens: int = 512,
) -> tuple[str, bool]:
    """
    Truncate text to max_tokens using the tokenizer.

    Returns:
        truncated_text: str
        was_truncated: bool
    """
    if not text:
        return "", False

    encoded = tokenizer(
        text,
        add_special_tokens=True,
        truncation=True,
        max_length=max_tokens,
        return_tensors=None,
    )

    was_truncated = len(encoded["input_ids"]) >= max_tokens

    truncated_text = tokenizer.decode(
        encoded["input_ids"],
        skip_special_tokens=True,
    )

    return truncated_text, was_truncated

In [ ]:
# =========================
# 2) PHASE 1: Download ALL news to JSONL
# =========================
def download_all_benzinga_news(fin: news_data.News, ticker: str, out_path: Path):
    date_ranges = generate_yearly_ranges(GLOBAL_DATE_FROM, GLOBAL_DATE_TO)

    total_raw = 0
    total_written = 0

    with open(out_path, "w", encoding="utf-8") as f_out:
        for date_from, date_to in date_ranges:
            print(f"  [Benzinga] {ticker}: {date_from}..{date_to}")
            page = PAGE_START

            while True:
                if page > PAGE_MAX:
                    print(f"    Page limit ({PAGE_MAX}) reached; stop this range.")
                    break

                try:
                    articles = fin.news(
                        company_tickers=ticker,
                        date_from=date_from,
                        date_to=date_to,
                        display_output="full",
                        pagesize=PAGE_SIZE,
                        page=page,
                    )
                except BadRequestError:
                    print(f"    BadRequestError at page={page}; stop this range.")
                    break

                if not articles:
                    break

                for art in articles:
                    total_raw += 1
                    raw_body = art.get("body", "") or ""
                    clean_body = html_body_to_plain_text(raw_body)

                    # Remove quotes to keep JSONL robust
                    clean_body = clean_body.replace('"', "").replace("'", "")
                    title = (art.get("title") or "").replace('"', "").replace("'", "")

                    record = {
                        "created": art.get("created"),
                        "title": title,
                        "body": clean_body,
                    }
                    f_out.write(json.dumps(record, ensure_ascii=False) + "\n")
                    total_written += 1

                page += 1

    return total_raw, total_written

In [ ]:
# =========================
# 3) PHASE 2: FinBERT scoring (truncate rule)
# =========================
def finbert_score_jsonl(
    in_jsonl: Path,
    out_jsonl: Path,
    finbert_tokenizer,
    finbert_model,
    finbert_config,
):
    total_in = 0
    total_scored = 0
    total_ignored = 0
    total_truncated = 0

    with open(in_jsonl, "r", encoding="utf-8") as f_in, open(out_jsonl, "w", encoding="utf-8") as f_out:
        for line_no, line in enumerate(f_in, start=1):
            line = line.strip()
            if not line:
                continue
            total_in += 1

            try:
                record = json.loads(line)
            except json.JSONDecodeError:
                continue

            text = record.get("body", "") or ""
            if not text:
                continue

            text, was_truncated = truncate_to_512_tokens(
                text,
                finbert_tokenizer,
                max_tokens=MAX_TOKENS,
            )
            if was_truncated:
                total_truncated += 1

            inputs = finbert_tokenizer(text, return_tensors="pt", truncation=True, max_length=MAX_TOKENS)
            inputs = {k: v.to(DEVICE) for k, v in inputs.items()}

            try:
                with torch.no_grad():
                    outputs = finbert_model(**inputs)

                logits = outputs.logits
                probs_tensor = torch.nn.functional.softmax(logits, dim=-1)[0]

                probs = probs_tensor.detach().cpu().tolist()
                pred_id = int(torch.argmax(probs_tensor).item())
                label = finbert_config.id2label[pred_id]
                prob_dict = {finbert_config.id2label[i]: probs[i] for i in range(len(probs))}

                out_record = {
                    "created": record.get("created"),
                    "title": record.get("title"),
                    "label": label,
                    "probs": prob_dict,
                }
                f_out.write(json.dumps(out_record, ensure_ascii=False) + "\n")
                total_scored += 1

            except RuntimeError:
                total_ignored += 1

    return total_in, total_scored, total_ignored, total_truncated

In [ ]:
# =========================
# 4) PHASE 3: Build TS-with-sentiment CSV
# =========================
def build_ts_with_sentiment(ts_csv_path: Path, finbert_jsonl_path: Path, out_csv_path: Path):
    df_price = pd.read_csv(ts_csv_path)

    if "Date" not in df_price.columns:
        raise ValueError(f"No 'Date' column in {ts_csv_path}")
    if "Close" not in df_price.columns:
        raise ValueError(f"No 'Close' column in {ts_csv_path}")

    df_price = df_price[["Date", "Close"]]
    df_price["Date"] = pd.to_datetime(df_price["Date"], errors="coerce", utc=True)
    df_price["Date"] = df_price["Date"].dt.tz_convert(None)
    df_price = df_price.set_index("Date").sort_index()
    df_price.index = df_price.index.normalize()

    rows = []
    with open(finbert_jsonl_path, "r", encoding="utf-8") as f_in:
        for line_no, line in enumerate(f_in, start=1):
            line = line.strip()
            if not line:
                continue
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            created_raw = rec.get("created")
            probs = rec.get("probs", {})
            if not created_raw or not probs:
                continue

            created_dt = pd.to_datetime(created_raw, errors="coerce", utc=True)
            if pd.isna(created_dt):
                continue

            d = created_dt.date()
            rows.append({
                "date": d,
                "pos": float(probs.get("positive", 0.0)),
                "neg": float(probs.get("negative", 0.0)),
                "neu": float(probs.get("neutral", 0.0)),
            })

    df_sent = pd.DataFrame(rows)
    if df_sent.empty:
        raise ValueError(f"No sentiment rows from {finbert_jsonl_path}")

    df_sent["date"] = pd.to_datetime(df_sent["date"])
    df_daily = (
        df_sent.groupby("date")
        .agg(
            pos_mean=("pos", "mean"),
            neg_mean=("neg", "mean"),
            neut_mean=("neu", "mean"),
            news_count=("pos", "count"),
        )
        .sort_index()
    )

    # Restrict prices to sentiment range (as in your notebook)
    first_sent_date = df_daily.index.min()
    last_sent_date = df_daily.index.max()

    df_price = df_price[(df_price.index >= first_sent_date) & (df_price.index <= last_sent_date)]

    df_merged = df_price.join(df_daily, how="left")
    for col in ["pos_mean", "neg_mean", "neut_mean", "news_count"]:
        if col not in df_merged.columns:
            df_merged[col] = 0.0
    df_merged[["pos_mean", "neg_mean", "neut_mean", "news_count"]] = (
        df_merged[["pos_mean", "neg_mean", "neut_mean", "news_count"]].fillna(0.0)
    )

    df_final = df_merged[["Close", "pos_mean", "neut_mean", "neg_mean", "news_count"]]
    df_final.to_csv(out_csv_path, index=True)

    return len(df_final), first_sent_date, last_sent_date


In [ ]:
# =========================
# 5) RUN: all tickers in 3 phases
# =========================
#api_key = get_api_key()
fin = news_data.News("bz.TEQRT3RPQWKNI25GFXG4PU4ONAQCMITV")

model_name = "ProsusAI/finbert"
finbert_tokenizer = AutoTokenizer.from_pretrained(model_name)
finbert_model = AutoModelForSequenceClassification.from_pretrained(model_name).to(DEVICE)
finbert_config = AutoConfig.from_pretrained(model_name)
finbert_model.eval()

print(f"[INFO] Loaded FinBERT: {model_name}")



In [ ]:
# -------- Phase 1 --------
print("\n=========================")
print("PHASE 1: Download ALL Benzinga news")
print("=========================")

phase1_stats = {}
for ticker in TICKERS:
    out_jsonl = BENZINGA_DIR / f"{ticker}.jsonl"
    print(f"\n[PHASE 1] {ticker} -> {out_jsonl.name}")

    raw_n, written_n = download_all_benzinga_news(fin, ticker, out_jsonl)
    phase1_stats[ticker] = {"raw": raw_n, "written": written_n}
    print(f"  [DONE] raw={raw_n} | written={written_n}")

In [ ]:
# -------- Phase 2 --------
print("\n=========================")
print("PHASE 2: FinBERT scoring (truncate to first 512 tokens if >512 tokens)")
print("=========================")

phase2_stats = {}
for ticker in TICKERS:
    in_jsonl = BENZINGA_DIR / f"{ticker}.jsonl"
    out_jsonl = FINBERT_DIR / f"{ticker}_finbert_sentiment.jsonl"

    print(f"\n[PHASE 2] {ticker} -> {out_jsonl.name}")

    if not in_jsonl.is_file():
        print(f"  [WARN] Missing input: {in_jsonl.as_posix()} -> skip")
        continue

    total_in, scored, ignored, truncated = finbert_score_jsonl(
        in_jsonl=in_jsonl,
        out_jsonl=out_jsonl,
        finbert_tokenizer=finbert_tokenizer,
        finbert_model=finbert_model,
        finbert_config=finbert_config,
    )

    phase2_stats[ticker] = {
        "in": total_in, "scored": scored, "ignored": ignored, "truncated": truncated
    }
    print(f"  [DONE] in={total_in} | scored={scored} | ignored={ignored} | truncated={truncated}")



In [ ]:
# -------- Phase 3 --------
print("\n=========================")
print("PHASE 3: Build ts_with_sentiment CSVs")
print("=========================")

phase3_statsison = {}
for ticker in TICKERS:
    ts_csv = TIME_SERIES_DIR / f"{ticker.lower()}.csv"
    fin_jsonl = FINBERT_DIR / f"{ticker}_finbert_sentiment.jsonl"
    out_csv = TS_WITH_SENTIMENT_DIR / f"{ticker}_ts_with_sentiment.csv"

    print(f"\n[PHASE 3] {ticker} -> {out_csv.name}")

    if not ts_csv.is_file():
        print(f"  [WARN] Missing TS CSV: {ts_csv.as_posix()} -> skip")
        continue
    if not fin_jsonl.is_file():
        print(f"  [WARN] Missing FinBERT JSONL: {fin_jsonl.as_posix()} -> skip")
        continue

    try:
        n_rows, d0, d1 = build_ts_with_sentiment(ts_csv, fin_jsonl, out_csv)
        print(f"  [DONE] rows={n_rows} | range={d0.date()}..{d1.date()} | saved={out_csv.as_posix()}")
    except Exception as e:
        print(f"  [ERROR] {ticker}: {e}")

print("\n[DONE] All phases finished.")